# Ranking Evaluation of Trained Models

This notebook loads previously trained models and evaluates them on the ranking task.
Computed metrics: `roc_auc`, `mrr`, `ndcg_at_5`, `ndcg_at_10`, `ndcg_at_50`, `ndcg_at_100`.

In [1]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

'1'

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR   = (PROJECT_ROOT / 'data' / 'link_prediction' / 'llm_concept_datasets').resolve()
OUTPUT_DIR = (PROJECT_ROOT / 'data' / 'link_prediction' / 'llm_concept_results').resolve()
MODEL_DIR  = OUTPUT_DIR / 'models'

RANKING_CSV = DATA_DIR / 'ranking_test.csv'
TRAIN_CSV   = DATA_DIR / 'train.csv'
VAL_CSV     = DATA_DIR / 'val.csv'

K_VALUES = [5, 10, 50, 100]

assert RANKING_CSV.is_file(), f'Missing {RANKING_CSV}'
assert TRAIN_CSV.is_file(),   f'Missing {TRAIN_CSV}'
assert MODEL_DIR.is_dir(),    f'Missing {MODEL_DIR}'

print('Checkpoints found:')
for p in sorted(MODEL_DIR.iterdir()):
    print(' ', p.name)


Checkpoints found:
  lightgbm_all.pkl
  mlp_all.pt
  mlp_emb.pt
  mlp_structure.pt
  xgboost_all.pkl


In [3]:
import numpy as np
import pandas as pd

from src.link_prediction.calculate_metrics import (
    compute_ranking_metrics,
    format_ranking_metrics,
)
from src.link_prediction.load_data import (
    classify_features,
    _select_cols,
    load_datasets,
)


## Model loading

In [4]:
# Map: model display name -> (checkpoint stem, kind, feature_groups)
MODEL_REGISTRY = [
    ('MLP (structure)', 'mlp_structure', 'mlp',  'structure'),
    ('MLP (emb)',       'mlp_emb',       'mlp',  'emb'),
    ('MLP (all)',       'mlp_all',       'mlp',  ['structure', 'emb']),
    ('LightGBM (all)',  'lightgbm_all',  'lgbm', 'all'),
    ('XGBoost (all)',   'xgboost_all',   'xgb',  'all'),
]

import json

def _fg_key(fg):
    return json.dumps(fg, sort_keys=True) if isinstance(fg, list) else fg

# Build the set of unique feature groups we need scalers for
needed_fgs = {}
for _, _, _, fg in MODEL_REGISTRY:
    needed_fgs[_fg_key(fg)] = fg

# Fit each scaler via load_datasets (which internally fits the scaler on train)
scaler_by_fg = {}
for key, fg in needed_fgs.items():
    ds = load_datasets(str(TRAIN_CSV), str(VAL_CSV), str(RANKING_CSV), fg)
    scaler_by_fg[key] = ds.scaler
    print(f'feature group {fg} -> n_features = {ds.train_ds.X.shape[1]}')


feature group structure -> n_features = 41
feature group emb -> n_features = 2
feature group ['structure', 'emb'] -> n_features = 43
feature group all -> n_features = 43


In [5]:
from src.link_prediction.models.mlp import MLPTrainer
from src.link_prediction.models.boosting_models import LightGBMModel, XGBoostModel

loaded_models = {}   # name -> (predict_callable, kind, fg)

for name, stem, kind, fg in MODEL_REGISTRY:
    if kind == 'mlp':
        path = MODEL_DIR / f'{stem}.pt'
    else:
        path = MODEL_DIR / f'{stem}.pkl'

    print(f'LOAD  {name} - {path.name}')
    try:
        if kind == 'mlp':
            obj = MLPTrainer.load(str(path))
            predict = lambda X, _o=obj: _o.predict(X)
        elif kind == 'lgbm':
            obj = LightGBMModel.load(str(path))
            predict = lambda X, _o=obj: _o.predict(X)
        elif kind == 'xgb':
            obj = XGBoostModel.load(str(path))
            predict = lambda X, _o=obj: _o.predict(X)
        loaded_models[name] = (predict, kind, fg)
        print(f'   OK ({name})')
    except Exception as e:
        print(f'   FAILED to load {name}: {type(e).__name__}: {e}')

print('\nLoaded models:', list(loaded_models.keys()))


LOAD  MLP (structure) - mlp_structure.pt
   OK (MLP (structure))
LOAD  MLP (emb) - mlp_emb.pt
   OK (MLP (emb))
LOAD  MLP (all) - mlp_all.pt
   OK (MLP (all))
LOAD  LightGBM (all) - lightgbm_all.pkl
   OK (LightGBM (all))
LOAD  XGBoost (all) - xgboost_all.pkl
   OK (XGBoost (all))

Loaded models: ['MLP (structure)', 'MLP (emb)', 'MLP (all)', 'LightGBM (all)', 'XGBoost (all)']


## Test set preparing

In [6]:
df_rank = pd.read_csv(RANKING_CSV)
print('ranking_test.csv:', df_rank.shape)

y_rank = df_rank['label'].values.astype(np.int32)
print('positives:', int(y_rank.sum()), '/', len(y_rank))

# Precompute scaled feature matrices, one per unique feature group
groups = classify_features(df_rank.columns.tolist())

X_by_fg = {}
for key, fg in needed_fgs.items():
    cols = _select_cols(groups, fg)
    if not cols:
        print(f'WARN: no columns for feature group {fg}')
        continue
    X = np.nan_to_num(df_rank[cols].values.astype(np.float32))
    X = scaler_by_fg[key].transform(X).astype(np.float32)
    X_by_fg[key] = X
    print(f'feature group {fg} -> X shape {X.shape}')


ranking_test.csv: (291739, 49)
positives: 1552 / 291739
feature group structure -> X shape (291739, 41)
feature group emb -> X shape (291739, 2)
feature group ['structure', 'emb'] -> X shape (291739, 43)
feature group all -> X shape (291739, 43)


## Model evaluations

In [10]:
import warnings
warnings.filterwarnings('ignore')

In [11]:
rank_results = {}   # name -> metrics dict

for name, (predict, kind, fg) in loaded_models.items():
    key = _fg_key(fg)
    if key not in X_by_fg:
        continue

    X = X_by_fg[key]

    try:
        scores = predict(X)
    except Exception as e:
        continue

    scores = np.asarray(scores).ravel()
    if scores.shape[0] != y_rank.shape[0]:
        continue

    metrics = compute_ranking_metrics(y_rank, scores, K_VALUES)
    rank_results[name] = metrics


In [12]:
wanted_cols = ['model', 'roc_auc', 'mrr',
               'ndcg_at_5', 'ndcg_at_10', 'ndcg_at_50', 'ndcg_at_100']

rows = []
for name, m in rank_results.items():
    row = {'model': name}
    for c in wanted_cols[1:]:
        row[c] = m.get(c)
    rows.append(row)

summary_df = pd.DataFrame(rows, columns=wanted_cols)

summary_df


,model,roc_auc,mrr,ndcg_at_5,ndcg_at_10,ndcg_at_50,ndcg_at_100
0,MLP (structure),0.887623,0.100000,0.000000,0.063621,0.183577,0.210412
1,MLP (emb),0.850443,0.023256,0.000000,0.000000,0.014202,0.024241
2,MLP (all),0.907381,0.250000,0.146068,0.094788,0.297210,0.296935
3,LightGBM (all),0.862720,0.016667,0.000000,0.000000,0.000000,0.008053
4,XGBoost (all),0.890436,0.500000,0.213986,0.202483,0.088240,0.054354
